# Set up

## Change working directory

In [1]:
%cd ../

C:\Users\Skylar\Downloads\tools


## Install packages

In [2]:
# Placeholder

## Import development libraries

In [3]:
import ipytest
import pytest
from sklearn import datasets as snds

ipytest.autoconfig(rewrite_asserts=False)

## Import other libraries

In [4]:
import typing

import networkx as nx
import numpy as np
import pandas as pd
import toolz as tz
from sklearn import base as snbe
from tools import data_analysis as tsda
from tools import utils as tsus

## Declare constants

In [5]:
module: str = "networkx"

# Define get_correlated_groups

## Define

In [6]:
def get_correlated_groups(correlations: pd.Series, thresholds: np.ndarray | None = None) -> pd.DataFrame:
    def set_interval_index(data: pd.DataFrame) -> pd.DataFrame:
        breaks: list[float] = data.index.tolist() + [1]
        return data.set_axis(labels=pd.IntervalIndex.from_breaks(breaks=breaks, closed="left"))

    if thresholds is None:
        thresholds: np.ndarray = np.arange(start=1e-2, stop=1, step=1e-2)
    correlated_groups: dict = {}
    for threshold in thresholds:  # type: float
        filtered_correlations: pd.DataFrame = (
            correlations.abs()
            .pipe(func=lambda x: x[x.ge(other=threshold)])
            .reset_index()
            .set_axis(labels=["source", "target", "r"], axis=1)
        )
        correlated_groups[threshold] = tz.pipe(
            filtered_correlations, tz.partial(nx.from_pandas_edgelist, edge_attr="r"), nx.connected_components, list
        )
    assign_args: dict[str, typing.Callable] = {
        "n_groups": lambda x: x["groups"].apply(func=len),
        "group_sizes": lambda x: x["groups"].apply(func=lambda x: list(map(len, x))),
        "min_group_sizes": lambda x: x["group_sizes"].apply(func=min),
        "max_group_sizes": lambda x: x["group_sizes"].apply(func=max),
        "total_group_sizes": lambda x: x["group_sizes"].apply(func=sum),
        "n_features_dropped": lambda x: x["total_group_sizes"].sub(other=x["n_groups"]),
    }
    return (
        pd.Series(data=correlated_groups)
        .drop_duplicates()
        .pipe(func=lambda x: x[x.apply(func=len).gt(other=1)])
        .to_frame(name="groups")
        .assign(**assign_args)
        .pipe(func=set_interval_index)
    )

## Demonstrate

In [7]:
# Read in data
X, y = data = snds.load_breast_cancer(return_X_y=True, as_frame=True)  # type: tuple[pd.DataFrame, pd.Series]
tsus.describe_structure(x=data)
# Get correlations
correlations: pd.Series = tsda.get_correlations(X=X)
print("-" * int(8e1), correlations, "-" * int(8e1), sep="\n")
# Get correlated groups
correlated_groups: pd.DataFrame = get_correlated_groups(correlations=correlations)
correlated_groups

<class 'tuple'> with 2 elements
- element: 0
  (<class 'pandas.core.frame.DataFrame'>, (569, 30))
- element: 1
  (<class 'pandas.core.series.Series'>, (569,))
--------------------------------------------------------------------------------
mean area         mean fractal dimension   -0.358425
mean radius       mean fractal dimension   -0.349931
mean area         smoothness error         -0.327431
mean radius       smoothness error         -0.326385
smoothness error  worst area               -0.323724
                                              ...   
worst radius      worst perimeter           0.993548
mean perimeter    mean area                 0.997068
mean radius       mean perimeter            0.997802
worst radius      worst area                0.998891
mean radius       mean area                 0.999602
Name: spearman, Length: 435, dtype: float64
--------------------------------------------------------------------------------


,groups,n_groups,group_sizes,min_group_sizes,max_group_sizes,total_group_sizes,n_features_dropped
"[0.45, 0.47000000000000003)","[{worst area, mean perimeter, worst fractal di...",2,"[27, 3]",3,27,30,28
"[0.47000000000000003, 0.48000000000000004)","[{symmetry error, smoothness error}, {worst ar...",3,"[2, 25, 3]",2,25,30,27
"[0.48000000000000004, 0.5)","[{worst area, worst fractal dimension, mean pe...",2,"[25, 3]",3,25,28,26
"[0.5, 0.56)","[{worst area, mean perimeter, worst fractal di...",2,"[25, 2]",2,25,27,25
"[0.56, 0.68)","[{worst area, worst fractal dimension, mean pe...",3,"[23, 2, 2]",2,23,27,24
"[0.68, 0.7100000000000001)","[{worst area, mean perimeter, worst fractal di...",4,"[21, 2, 2, 2]",2,21,27,23
"[0.7100000000000001, 0.72)","[{mean symmetry, worst symmetry}, {worst area,...",4,"[2, 21, 2, 2]",2,21,27,23
"[0.72, 0.77)","[{worst area, mean perimeter, worst fractal di...",3,"[21, 2, 2]",2,21,25,22
"[0.77, 0.78)","[{worst area, mean perimeter, worst compactnes...",3,"[19, 2, 2]",2,19,23,20
"[0.78, 0.79)","[{worst area, worst perimeter, worst concave p...",4,"[16, 2, 2, 3]",2,16,23,19


## Test

In [8]:
# Placeholder

# Define get_correlated_features_to_drop

## Define

In [9]:
def get_correlated_features_to_drop(
    correlated_feature_groups: list[set[str]], correlations_with_target: pd.Series, print_shapes: bool = True
) -> list[str]:
    features_to_drop: list[str] = []
    for correlated_feature_group in correlated_feature_groups:  # type: set[str]
        these_correlations_with_target: pd.Series = correlations_with_target.loc[list(correlated_feature_group)]
        best_feature: str = these_correlations_with_target.abs().idxmax()
        remaining_features: set[str] = correlated_feature_group.difference([best_feature])
        features_to_drop.extend(remaining_features)
        if print_shapes:
            tsus.print_shapes(x=[correlated_feature_groups, remaining_features, features_to_drop], sep=" -> ")
    return features_to_drop

## Demonstrate

In [10]:
# Get correlated feature groups
correlated_feature_groups: list[set[str]] = correlated_groups.loc[9.9e-1, "groups"]
print(correlated_feature_groups, end="\n" * 2)
# Get correlations with target
correlations_with_target: pd.Series = tsda.get_correlations(X=X, y=y)
tz.pipe(
    correlated_feature_groups,
    tz.curried.map(lambda x: correlations_with_target.loc[list(x)].sort_values()),
    lambda x: print(*x, sep="\n" * 2, end="\n" * 2),
)
# Drop correlated features with lowest correlations with target
sorted(
    get_correlated_features_to_drop(
        correlated_feature_groups=correlated_feature_groups, correlations_with_target=correlations_with_target
    )
)

[{'worst perimeter', 'worst area', 'worst radius'}, {'mean radius', 'mean perimeter', 'mean area'}]

worst perimeter   -0.796319
worst radius      -0.787933
worst area        -0.786902
Name: spearman, dtype: float64

mean perimeter   -0.748496
mean area        -0.734122
mean radius      -0.732785
Name: spearman, dtype: float64

2 -> 2 -> 2
2 -> 2 -> 4


['mean area', 'mean radius', 'worst area', 'worst radius']

## Test

In [11]:
# Placeholder

# Define CorrelatedDropper

## Define

In [12]:
class CorrelatedDropper(snbe.BaseEstimator, snbe.TransformerMixin):
    def __init__(self, is_memory_low: bool = False, threshold: float = 1e0) -> None:
        self.is_memory_low, self.threshold = is_memory_low, threshold

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "CorrelatedDropper":
        self.correlations_between_features: pd.DataFrame = tz.pipe(
            X, tz.partial(tsda.get_correlations, is_memory_low=self.is_memory_low), get_correlated_groups
        )
        self.correlations_with_target: pd.Series = tsda.get_correlations(X=X, y=y, is_memory_low=self.is_memory_low)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        first_left_inclusive_bound: float = self.correlations_between_features.index[0].left
        last_right_exclusive_bound: float = self.correlations_between_features.index[-1].right
        if self.threshold >= last_right_exclusive_bound:
            features_to_drop: list[str] = []
        else:
            self.updated_threshold: float = max(self.threshold, first_left_inclusive_bound)
            features_to_drop: list[str] = get_correlated_features_to_drop(
                correlated_feature_groups=self.correlations_between_features.loc[self.updated_threshold, "groups"],
                correlations_with_target=self.correlations_with_target,
                print_shapes=False,
            )
        return X.drop(columns=features_to_drop)

    def get_feature_names_out():
        pass

## Demonstrate

In [13]:
print("Shape:", X.shape, end=" -> ")
dropper = CorrelatedDropper(threshold=7e-1)
X_t: pd.DataFrame = dropper.fit_transform(X=X, y=y)
print(X_t.shape)
display(X_t)
dropper.correlations_between_features.loc[dropper.updated_threshold, :]

Shape: (569, 30) -> (569, 7)


,texture error,smoothness error,symmetry error,worst texture,worst perimeter,worst smoothness,worst symmetry
0,0.9053,0.006399,0.03003,17.33,184.60,0.16220,0.4601
1,0.7339,0.005225,0.01389,23.41,158.80,0.12380,0.2750
2,0.7869,0.006150,0.02250,25.53,152.50,0.14440,0.3613
3,1.1560,0.009110,0.05963,26.50,98.87,0.20980,0.6638
4,0.7813,0.011490,0.01756,16.67,152.20,0.13740,0.2364
...,...,...,...,...,...,...,...
564,1.2560,0.010300,0.01114,26.40,166.10,0.14100,0.2060
565,2.4630,0.005769,0.01898,38.25,155.00,0.11660,0.2572
566,1.0750,0.005903,0.01318,34.12,126.70,0.11390,0.2218
567,1.5950,0.006522,0.02324,39.42,184.60,0.16500,0.4087


groups                [{worst area, mean perimeter, worst fractal di...
n_groups                                                              4
group_sizes                                               [21, 2, 2, 2]
min_group_sizes                                                       2
max_group_sizes                                                      21
total_group_sizes                                                    27
n_features_dropped                                                   23
Name: [0.68, 0.7100000000000001), dtype: object

## Test

In [14]:
%%ipytest

scenario_mappings: dict[str, float] = {
    'below_min': 4e-1,
    'above_max': 1e0,
    'in_between': 7e-1
}

# Property-based test(s)
@pytest.mark.parametrize('scenario', scenario_mappings.keys())
def test_correlated_dropper(scenario: str) -> None:
    X, y = data = snds.load_breast_cancer(return_X_y=True, as_frame=True)  # type: tuple[pd.DataFrame, pd.Series]
    threshold: float = scenario_mappings[scenario]
    dropper = CorrelatedDropper(threshold=threshold).fit(*data)
    out: pd.DataFrame = dropper.transform(X=X)
    # Property 1: Output is DataFrame
    assert isinstance(out, pd.DataFrame)
    # Property 2: No rows are dropped
    assert X.shape[0] == out.shape[0]
    # Property 3: Number of columns dropped is correct
    if scenario == 'above_max':
        assert X.shape[1] == out.shape[1] # Nothing dropped
    else:
        updated_threshold: float = dropper.updated_threshold
        if scenario == 'below_min':
            assert updated_threshold > dropper.threshold
        n_features_dropped: int = dropper.correlations_between_features.loc[updated_threshold, 'n_features_dropped']
        assert out.shape[1] == X.shape[1] - n_features_dropped

...                                                                                          [100%]
3 passed in 0.80s


# Write module

## Write

In [15]:
module_path: str = "src/tools/%s.py" % module

In [16]:
%%file $module_path

import typing

import networkx as nx
import numpy as np
import pandas as pd
import toolz as tz
from sklearn import base as snbe
from tools import data_analysis as tsda
from tools import utils as tsus

class CorrelatedDropper(snbe.BaseEstimator, snbe.TransformerMixin):
    def __init__(self, is_memory_low: bool = False, threshold: float = 1e0) -> None:
        self.is_memory_low, self.threshold = is_memory_low, threshold

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "CorrelatedDropper":
        self.correlations_between_features: pd.DataFrame = tz.pipe(
            X, tz.partial(tsda.get_correlations, is_memory_low=self.is_memory_low), get_correlated_groups
        )
        self.correlations_with_target: pd.Series = tsda.get_correlations(X=X, y=y, is_memory_low=self.is_memory_low)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        first_left_inclusive_bound: float = self.correlations_between_features.index[0].left
        last_right_exclusive_bound: float = self.correlations_between_features.index[-1].right
        if self.threshold >= last_right_exclusive_bound:
            features_to_drop: list[str] = []
        else:
            self.updated_threshold: float = max(self.threshold, first_left_inclusive_bound)
            features_to_drop: list[str] = get_correlated_features_to_drop(
                correlated_feature_groups=self.correlations_between_features.loc[self.updated_threshold, "groups"],
                correlations_with_target=self.correlations_with_target,
                print_shapes=False,
            )
        return X.drop(columns=features_to_drop)

    def get_feature_names_out():
        pass

def get_correlated_features_to_drop(
    correlated_feature_groups: list[set[str]], correlations_with_target: pd.Series, print_shapes: bool = True
) -> list[str]:
    features_to_drop: list[str] = []
    for correlated_feature_group in correlated_feature_groups:  # type: set[str]
        these_correlations_with_target: pd.Series = correlations_with_target.loc[list(correlated_feature_group)]
        best_feature: str = these_correlations_with_target.abs().idxmax()
        remaining_features: set[str] = correlated_feature_group.difference([best_feature])
        features_to_drop.extend(remaining_features)
        if print_shapes:
            tsus.print_shapes(x=[correlated_feature_groups, remaining_features, features_to_drop], sep=" -> ")
    return features_to_drop

def get_correlated_groups(correlations: pd.Series, thresholds: np.ndarray | None = None) -> pd.DataFrame:
    def set_interval_index(data: pd.DataFrame) -> pd.DataFrame:
        breaks: list[float] = data.index.tolist() + [1]
        return data.set_axis(labels=pd.IntervalIndex.from_breaks(breaks=breaks, closed="left"))

    if thresholds is None:
        thresholds: np.ndarray = np.arange(start=1e-2, stop=1, step=1e-2)
    correlated_groups: dict = {}
    for threshold in thresholds:  # type: float
        filtered_correlations: pd.DataFrame = (
            correlations.abs()
            .pipe(func=lambda x: x[x.ge(other=threshold)])
            .reset_index()
            .set_axis(labels=["source", "target", "r"], axis=1)
        )
        correlated_groups[threshold] = tz.pipe(
            filtered_correlations, tz.partial(nx.from_pandas_edgelist, edge_attr="r"), nx.connected_components, list
        )
    assign_args: dict[str, typing.Callable] = {
        "n_groups": lambda x: x["groups"].apply(func=len),
        "group_sizes": lambda x: x["groups"].apply(func=lambda x: list(map(len, x))),
        "min_group_sizes": lambda x: x["group_sizes"].apply(func=min),
        "max_group_sizes": lambda x: x["group_sizes"].apply(func=max),
        "total_group_sizes": lambda x: x["group_sizes"].apply(func=sum),
        "n_features_dropped": lambda x: x["total_group_sizes"].sub(other=x["n_groups"]),
    }
    return (
        pd.Series(data=correlated_groups)
        .drop_duplicates()
        .pipe(func=lambda x: x[x.apply(func=len).gt(other=1)])
        .to_frame(name="groups")
        .assign(**assign_args)
        .pipe(func=set_interval_index)
    )

Overwriting src/tools/networkx.py


## Format

In [17]:
!ruff format $module_path

1 file reformatted


## Check with ruff

In [18]:
!ruff check $module_path

All checks passed!


## Check with ty

In [19]:
!ty check $module_path

error[invalid-assignment]: Object of type `Hashable` is not assignable to `str`
  --> src\tools\networkx.py:47:23
   |
47 |         best_feature: str = these_correlations_with_target.abs().idxmax()
   |                       ---   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Incompatible value of type `Hashable`
   |                       |
   |                       Declared type
   |

error[no-matching-overload]: No overload of function `arange` matches arguments
  --> src\tools\networkx.py:61:34
   |
61 |         thresholds: np.ndarray = np.arange(start=1e-2, stop=1, step=1e-2)
   |                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
   |
info: First overload defined here
   --> env\Lib\site-packages\numpy\_core\multiarray.pyi:968:1
    |
968 | / @overload  # dtype=<known>
969 | | def arange(
970 | |     start_or_stop: _ArangeScalar | float,
971 | |     /,
972 | |     stop: _ArangeScalar | float | None = None,
973 | |     step: _ArangeScalar | float | None = 1